# 3.1.5, Cartan / SN derivator identities (form + dual multivector)

**Goal.** Like the three derived identities in 3.1.4, but this time
in the *derivator-of-derivation* shape: for a derivation $D$ paired
with a bracket $[\cdot,\cdot]$, the *derivator* obstruction
$D^{[\cdot,\cdot]}_D(a,b) := D[a,b] - [Da, b] - (-1)^{|a||D|}[a, Db]$
gives rise to six canonical configurations that are not identically
zero (three with the Koszul bracket on the form side, three with the
Schouten–Nijenhuis bracket on the multivector side); we close all six
syntactically:

$$
\begin{aligned}
\textbf{Form } &\\
(1)\;& D^{T^{*}M}_{\mathcal{L}_U}(\eta, \mu)
       \;=\; \mathcal{L}_{\tilde{K}_\eta U}\,\mu + K_{\tilde{K}_\mu U}\,\eta, \\
(2)\;& \mathcal{L}_{\tilde{d}\,\tilde{\iota}_\eta U}\,\mu
       \;=\; -\,[d\,\iota_U \eta,\;\mu]_K, \\
(3)\;& 0 \;=\; d\,\iota_{\tilde{\mathcal{L}}_\omega W}\,\eta
       - d\,\iota_{\tilde{d}\,\tilde{\iota}_\eta W}\,\omega
       + d\,\iota_W\,[\omega, \eta]_K. \\[4pt]
\textbf{Dual (multivector) } &\\
(1')\;& \tilde{D}^{\mathrm{SN}}_{\tilde{\mathcal{L}}_\eta}(U, V)
       \;=\; \tilde{\mathcal{L}}_{K_U \eta}\,V + \tilde{K}_{K_V \eta}\,U, \\
(2')\;& \tilde{\mathcal{L}}_{d\,\iota_U \eta}\,V
       \;=\; -\,[\tilde{d}\,\tilde{\iota}_\eta U,\;V]_{\mathrm{SN}}, \\
(3')\;& 0 \;=\; \tilde{d}\,\tilde{\iota}_{\mathcal{L}_U \eta}\,V
       - \tilde{d}\,\tilde{\iota}_{d\,\iota_V \eta}\,U
       + \tilde{d}\,\tilde{\iota}_\eta\,[U, V]_{\mathrm{SN}}.
\end{aligned}
$$

The six identities of Tu/Cattaneo §3.1.5, paired as
$(1)\leftrightarrow(1')$, $(2)\leftrightarrow(2')$, $(3)\leftrightarrow(3')$, concretise the
form/multivector duality between the Koszul and Schouten–Nijenhuis
brackets. All proofs run under the Poisson assumption
($[\pi,\pi]_{\mathrm{SN}} = 0$).


## Strategy

**Form side (1)/(2)/(3).** `KoszulProblem.derivator_form_engine()`
, `intrinsic_engine_with_closure` baseline + form-side Cartan
intrinsics + 10 closure axioms (Phase 15.C pass 1), closes via
`prove_derivator(…, side="form")` under a `MultiEval` wrapper
evaluated against a vector-field probe $Y$.

**Multivector side (1')/(2')/(3').**
`derivator_multivector_engine()` is the dual extension of the
form-side engine: 13 closure axioms (Phase 15.C pass 2 + pass 3) are
added to the same baseline:

* SN→LBVF coercion (for 1-vfs); tilde-iota / pairing-flip /
  d-on-scalar / L-on-scalar bridges; HVF inner-iota normalisation;
  SN/LBVF Neg and Sum linearities (pass 2),
* `TildeLieOnOneVectorLichnerowiczDefinition`, the Lichnerowicz
  expansion $\tilde{\mathcal{L}}_\alpha U = -X_{\alpha(U)} + \pi^\sharp(\mathcal{L}_U \alpha)
  - [U, \pi^\sharp \alpha]_{VF}$,
* `SharpNegLinearityDefinition`, $\pi^\sharp(\mathrm{Neg}(A)) \to
  \mathrm{Neg}(\pi^\sharp(A))$ so that signs can move outside, pair
  up, and cancel,
* `BareAtomSlotLiftDefinition`, special slots of opaque operator
  atoms like `LieBracketVF` are also opened by the inner engine when
  the atom is a direct `MultiEval` argument (pass 3).

`prove_derivator(…, side="multivector")` evaluates against a
1-form probe $\xi$ (`slot_kind="covector"`). The roles of Y and
U/V/W are settled inside `KoszulProblem`: Y, Z probes are
`Derivation` atoms, while U, V, W are registry-declared 1-vector
`Symbol`s, this lets `_is_plain_vf` and the closure axioms
recognise them correctly.


In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401


## 1. Setup

We exercise both the form and the multivector side from a single
`KoszulProblem`: the form inventory $(\omega, \eta, \mu)$ as
1-forms + the multivector inventory $(U, V, W)$ as 1-vfs. Probe
variables:

* $Y$, $Z$, `Derivation` atoms (form-side `MultiEval` evaluation),
* $\xi$, $\zeta$, registry-declared 1-form `Symbol`s
  (multivector-side `MultiEval` evaluation).

`prob.assume_poisson()` posits the Jacobi condition; the tilde-side
rules ($\tilde{d}^2 = 0$, $\tilde{\mathcal{L}}_\pi = 0$, etc.)
are gated on this assumption.


In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.brackets.base import BracketApply
from jacopy.brackets.schouten import sn as default_sn
from jacopy.calculus.cartan_remainder import K
from jacopy.calculus.derivator import derivator
from jacopy.calculus.exterior_d import d as default_d
from jacopy.calculus.interior import interior
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.calculus.tilde import K_tilde, tilde_d, tilde_interior, tilde_lie
from jacopy.core.expr import Neg, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.display.jupyter import display_chain
from jacopy.library.koszul_problem import KoszulProblem


reg = PropertyRegistry()
pi = Symbol("π")
omega = Symbol("ω"); reg.declare(omega, Graded(degree=1))
eta = Symbol("η");   reg.declare(eta, Graded(degree=1))
mu = Symbol("μ");    reg.declare(mu, Graded(degree=1))
U = Symbol("U");     reg.declare(U, Graded(degree=1))
V = Symbol("V");     reg.declare(V, Graded(degree=1))
W = Symbol("W");     reg.declare(W, Graded(degree=1))

Y = Derivation("Y", 0)
Z = Derivation("Z", 0)
xi = Symbol("ξ");    reg.declare(xi, Graded(degree=1))
zeta = Symbol("ζ");  reg.declare(zeta, Graded(degree=1))

prob = KoszulProblem(
    pi, (omega, eta, mu),
    registry=reg,
    multivectors=((U, 1), (V, 1), (W, 1)),
)
prob.assume_poisson()
K_b = prob.koszul_bracket

print("Form-side engine rule count       :",
      len(prob.derivator_form_engine().definitions))
print("Multivector-side engine rule count:",
      len(prob.derivator_multivector_engine().definitions))


Form-side engine rule count       : 33
Multivector-side engine rule count: 63


## 2. Form side

### (1) $D^{T^*M}_{\mathcal{L}_U}(\eta, \mu) = \mathcal{L}_{\tilde{K}_\eta U}\,\mu + K_{\tilde{K}_\mu U}\,\eta$

The left side is the obstruction of the derivation $\mathcal{L}_U$ on
the Koszul bracket $[\eta, \mu]_K$; on the right, two correction
terms tied to $\tilde{K}$ (the tilde Cartan-remainder). Evaluation
against $Y$ drops everything to a scalar.


In [3]:
lhs = derivator(lie_derivative(U), K_b, eta, mu)
K_tilde_eta_U = Act(K_tilde(eta, pi), U)
K_tilde_mu_U  = Act(K_tilde(mu, pi),  U)
rhs = Sum(
    Act(lie_derivative(K_tilde_eta_U), mu),
    Act(K(K_tilde_mu_U), eta),
)
chain = prob.prove_derivator(lhs, rhs, eval_args=(Y,), side="form")
print(f"(1) form-side, D^K_{{L_U}}(eta,mu) = L_{{Ktilde_eta U}} mu + K_{{Ktilde_mu U}} eta  -->  closed in {len(chain)} steps")
display_chain(chain)


(1) form-side, D^K_{L_U}(eta,mu) = L_{Ktilde_eta U} mu + K_{Ktilde_mu U} eta  -->  closed in 109 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left[\eta,\, \mu\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\eta)\!\left(\mu\right) - L_\pi\sharp(\mu)\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \mu \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\langle \pi\sharp\!\left(\eta\right),\, \mu \rangle \to \mu\!\left(\pi\sharp\!\left(\eta\right)\right) \quad \text{[\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle} \ensuremath{\to} \ensuremath{\alpha}(X)]\,(axiom)} \\
\mu\!\left(\pi\sharp\!\left(\eta\right)\right) \to -\eta\!\left(\pi\sharp\!\left(\mu\right)\right) \quad \text{[\ensuremath{\alpha}(\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\beta})) \ensuremath{\to} \ensuremath{-}\ensuremath{\beta}(\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\alpha})) [\ensuremath{\pi}]]\,(axiom)} \\
\left[L_U\!\left(\eta\right),\, \mu\right]_{[\cdot,\cdot]_K[\pi]} \to L_{\pi\sharp(L_U(\eta))}\!\left(\mu\right) - L_\pi\sharp(\mu)\!\left(L_U\!\left(\eta\right)\right) - d\!\left(\langle \pi\sharp\!\left(L_U\!\left(\eta\right)\right),\, \mu \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\langle \pi\sharp\!\left(L_U\!\left(\eta\right)\right),\, \mu \rangle \to \mu\!\left(\pi\sharp\!\left(L_U\!\left(\eta\right)\right)\right) \quad \text{[\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle} \ensuremath{\to} \ensuremath{\alpha}(X)]\,(axiom)} \\
\mu\!\left(\pi\sharp\!\left(L_U\!\left(\eta\right)\right)\right) \to -\left(L_U\!\left(\eta\right)\right)\!\left(\pi\sharp\!\left(\mu\right)\right) \quad \text{[\ensuremath{\alpha}(\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\beta})) \ensuremath{\to} \ensuremath{-}\ensuremath{\beta}(\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\alpha})) [\ensuremath{\pi}]]\,(axiom)} \\
\left(L_U\!\left(\eta\right)\right)\!\left(\pi\sharp\!\left(\mu\right)\right) \to U\!\left(\eta\!\left(\pi\sharp\!\left(\mu\right)\right)\right) - \eta\!\left([U,\pi\sharp(\mu)]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left[\eta,\, L_U\!\left(\mu\right)\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\eta)\!\left(L_U\!\left(\mu\right)\right) - L_{\pi\sharp(L_U(\mu))}\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, L_U\!\left(\mu\right) \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\langle \pi\sharp\!\left(\eta\right),\, L_U\!\left(\mu\right) \rangle \to \left(L_U\!\left(\mu\right)\right)\!\left(\pi\sharp\!\left(\eta\right)\right) \quad \text{[\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle} \ensuremath{\to} \ensuremath{\alpha}(X)]\,(axiom)} \\
\left(L_U\!\left(\mu\right)\right)\!\left(\pi\sharp\!\left(\eta\right)\right) \to U\!\left(\mu\!\left(\pi\sharp\!\left(\eta\right)\right)\right) - 

In [4]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](η, μ)
 -> (L_π♯(η)(μ) + (-L_π♯(μ)(η)) + (-d(⟨π♯(η), μ⟩)))

[2] ⟨α, X⟩ → α(X) [axiom]
    ⟨π♯(η), μ⟩
 -> μ(π♯(η))

[3] α(π♯(β)) → −β(π♯(α)) [π] [axiom]
    μ(π♯(η))
 -> (-η(π♯(μ)))

[4] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](L_U(η), μ)
 -> (L_π♯(L_U(η))(μ) + (-L_π♯(μ)(L_U(η))) + (-d(⟨π♯(L_U(η)), μ⟩)))

[5] ⟨α, X⟩ → α(X) [axiom]
    ⟨π♯(L_U(η)), μ⟩
 -> μ(π♯(L_U(η)))

[6] α(π♯(β)) → −β(π♯(α)) [π] [axiom]
    μ(π♯(L_U(η)))
 -> (-L_U(η)(π♯(μ)))

[7] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_U(η)(π♯(μ))
 -> (U(η(π♯(μ))) + (-η([U,π♯(μ)]_VF)))

[8] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](η, L_U(μ))
 -> (L_π♯(η)(L_U(μ)) + (-L_π♯(L_U(μ))(η)) + (-d(⟨π♯(η), L_U(μ)⟩)))

[9] ⟨α, X⟩ → α(X) [axiom]
    ⟨π♯(η), L_U(μ)⟩
 -> L_U(μ)(π♯(η))

[10] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, 

### (2) $\mathcal{L}_{\tilde{d}\,\tilde{\iota}_\eta U}\,\mu = -[d\,\iota_U \eta,\;\mu]_K$

On the left, $\tilde{d}\,\tilde{\iota}_\eta U$ is a vector field
($\tilde{\iota}_\eta$ lowers SN-degree, $\tilde{d}_\pi$ raises it;
starting from a 1-vector $U$ we land in a 1-vector again). On the
right, $d\,\iota_U \eta$ is a 1-form, taken in the Koszul bracket
against $\mu$.


In [5]:
head_lhs = Act(tilde_d(pi), Act(tilde_interior(eta), U))
lhs = Act(lie_derivative(head_lhs), mu)
head_rhs = Act(default_d, Act(interior(U), eta))
rhs = Neg(BracketApply(K_b, head_rhs, mu))
chain = prob.prove_derivator(lhs, rhs, eval_args=(Y,), side="form")
print(f"(2) form-side, L_{{dtilde iotatilde_eta U}} mu = -[d iota_U eta, mu]_K  -->  closed in {len(chain)} steps")
display_chain(chain)


(2) form-side, L_{dtilde iotatilde_eta U} mu = -[d iota_U eta, mu]_K  -->  closed in 21 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(-L_{X_{\iota_U(\eta)}}\!\left(\mu\right)\right)\!\left(Y\right) \to -\left(L_{X_{\iota_U(\eta)}}\!\left(\mu\right)\right)\!\left(Y\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(L_{X_{\iota_U(\eta)}}\!\left(\mu\right)\right)\!\left(Y\right) \to X_{\iota_U(\eta)}\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([X_{\iota_{U(\eta),Y]_{VF}}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left[d\!\left(\iota_U\!\left(\eta\right)\right),\, \mu\right]_{[\cdot,\cdot]_K[\pi]} \to L_{\pi\sharp(d(\iota_U(\eta)))}\!\left(\mu\right) - L_\pi\sharp(\mu)\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right) - d\!\left(\langle \pi\sharp\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right),\, \mu \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
L_{\pi\sharp(d(\iota_U(\eta)))}\!\left(\mu\right) \to L_{X_{\iota_U(\eta)}}\!\left(\mu\right) \quad \text{[atom-slot lift]\,(axiom)} \\
\pi\sharp\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right) \to X_{\iota_U(\eta)} \quad \text{[\ensuremath{\pi}\ensuremath{\sharp}(df) = X\_f [\ensuremath{\pi}]]\,(axiom)} \\
\langle X_{\iota_U(\eta)},\, \mu \rangle \to -\left(\pi\sharp\!\left(\mu\right)\right)\!\left(\iota_U\!\left(\eta\right)\right) \quad \text{[\ensuremath{\langle}X\_f, \ensuremath{\mu}\ensuremath{\rangle} = \ensuremath{-}\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\mu})(f) [\ensuremath{\pi}]]\,(axiom)} \\
\left(\pi\sharp\!\left(\mu\right)\right)\!\left(\iota_U\!\left(\eta\right)\right) \to \left(\pi\sharp\!\left(\mu\right)\right)\!\left(\eta\!\left(U\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(-\left(L_{X_{\iota_U(\eta)}}\!\left(\mu\right) - L_\pi\sharp(\mu)\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right) - d\!\left(-\left(\pi\sharp\!\left(\mu\right)\right)\!\left(\eta\!\left(U\right)\right)\right)\right)\right)\!\left(Y\right) \to -\left(L_{X_{\iota_U(\eta)}}\!\left(\mu\right) - L_\pi\sharp(\mu)\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right) - d\!\left(-\left(\pi\sharp\!\left(\mu\right)\right)\!\left(\eta\!\left(U\right)\right)\right)\right)\!\left(Y\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(L_{X_{\iota_U(\eta)}}\!\left(\mu\right) - L_\pi\sharp(\mu)\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right) - d\!\left(-\left(\pi\sharp\!\left(\mu\right)\right)\!\left(\eta\!\left(U\right)\right)\right)\right)\!\left(Y\right) \to \left(L_{X_{\iota_U(\eta)}}\!\left(\mu\right)\right)\!\left(Y\right) - \left(L_\pi\sharp(\mu)\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right)\right)\!\left(Y\right) - \left(d\!\left(-\left(\pi\sharp\!\left(\mu\right)\right)\!\left(\eta\!\left(U\right)\right)\right)\right)\!\left(Y\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(L_{X_{\iota_U(\eta)}}\!\left(\mu\right)\right)\!\left(Y\right) \to X_{\iota_U(\eta)}\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([X_{\iota_{U(\eta),Y]_{VF}}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_\pi\sharp(\mu)\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right)\right)\!\left(Y\ri

In [6]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (-L_X_ι_U(η)(μ))(Y)
 -> (-L_X_ι_U(η)(μ)(Y))

[2] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X_ι_U(η)(μ)(Y)
 -> (X_ι_U(η)(μ(Y)) + (-μ([X_ι_U(η),Y]_VF)))

[3] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](d(ι_U(η)), μ)
 -> (L_π♯(d(ι_U(η)))(μ) + (-L_π♯(μ)(d(ι_U(η)))) + (-d(⟨π♯(d(ι_U(η))), μ⟩)))

[4] atom-slot lift [axiom]
    L_π♯(d(ι_U(η)))(μ)
 -> L_X_ι_U(η)(μ)

[5] π♯(df) = X_f [π] [axiom]
    π♯(d(ι_U(η)))
 -> X_ι_U(η)

[6] ⟨X_f, μ⟩ = −π♯(μ)(f) [π] [axiom]
    ⟨X_ι_U(η), μ⟩
 -> (-π♯(μ)(ι_U(η)))

[7] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    π♯(μ)(ι_U(η))
 -> π♯(μ)(η(U))

[8] MultiEval head linearity [axiom]
    (-(L_X_ι_U(η)(μ) + (-L_π♯(μ)(d(ι_U(η)))) + (-d((-π♯(μ)(η(U)))))))(Y)
 -> (-(L_X_ι_U(η)(μ) + (-L_π♯(μ)(d(ι_U(η)))) + (-d((-π♯(μ)(η(U))))))(Y))

[9] MultiEval head linearity [axiom]
    (L_X_ι_U(η)(μ) + (-L_π♯(μ)(d(ι_U(η)))) + (-d((-π♯(μ)(η(U)))

### (3) $0 = d\,\iota_{\tilde{\mathcal{L}}_\omega W}\,\eta - d\,\iota_{\tilde{d}\,\tilde{\iota}_\eta W}\,\omega + d\,\iota_W\,[\omega, \eta]_K$

A zero-sum identity; `prove_derivator` takes two `Expr`s, so we
group the first two terms as $\mathrm{lhs}_{\mathrm{pair}}$ and the
third (positive) as $\mathrm{rhs}_{\mathrm{third}}$, presented in
the form `lhs_pair = rhs_third`.


In [7]:
v1 = Act(tilde_lie(omega, pi), W)
v2 = Act(tilde_d(pi), Act(tilde_interior(eta), W))
bracket_oe = BracketApply(K_b, omega, eta)

lhs_pair = Sum(
    Act(default_d, Act(interior(v1), eta)),
    Act(default_d, Act(interior(W), bracket_oe)),
)
rhs_third = Act(default_d, Act(interior(v2), omega))
chain = prob.prove_derivator(lhs_pair, rhs_third, eval_args=(Y,), side="form")
print(f"(3) form-side, zero-sum reformulated lhs_pair = rhs_third  -->  closed in {len(chain)} steps")
display_chain(chain)


(3) form-side, zero-sum reformulated lhs_pair = rhs_third  -->  closed in 30 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left[\omega,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle \to \eta\!\left(\pi\sharp\!\left(\omega\right)\right) \quad \text{[\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle} \ensuremath{\to} \ensuremath{\alpha}(X)]\,(axiom)} \\
\left(d\!\left(-\iota_{X_{\iota_W(\omega)}}\!\left(\eta\right) + \left(\iota_{\pi\sharp(L_W(\omega))}\!\left(\eta\right) - \iota_{[W,\pi\sharp(\omega)]_{VF}}\!\left(\eta\right)\right)\right) + d\!\left(\iota_W\!\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\eta\!\left(\pi\sharp\!\left(\omega\right)\right)\right)\right)\right)\right)\!\left(Y\right) \to \left(d\!\left(-\iota_{X_{\iota_W(\omega)}}\!\left(\eta\right) + \left(\iota_{\pi\sharp(L_W(\omega))}\!\left(\eta\right) - \iota_{[W,\pi\sharp(\omega)]_{VF}}\!\left(\eta\right)\right)\right)\right)\!\left(Y\right) + \left(d\!\left(\iota_W\!\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\eta\!\left(\pi\sharp\!\left(\omega\right)\right)\right)\right)\right)\right)\!\left(Y\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(d\!\left(-\iota_{X_{\iota_W(\omega)}}\!\left(\eta\right) + \left(\iota_{\pi\sharp(L_W(\omega))}\!\left(\eta\right) - \iota_{[W,\pi\sharp(\omega)]_{VF}}\!\left(\eta\right)\right)\right)\right)\!\left(Y\right) \to Y\!\left(-\iota_{X_{\iota_W(\omega)}}\!\left(\eta\right) + \left(\iota_{\pi\sharp(L_W(\omega))}\!\left(\eta\right) - \iota_{[W,\pi\sharp(\omega)]_{VF}}\!\left(\eta\right)\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\iota_W\!\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\eta\!\left(\pi\sharp\!\left(\omega\right)\right)\right)\right)\right)\right)\!\left(Y\right) \to Y\!\left(\iota_W\!\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\eta\!\left(\pi\sharp\!\left(\omega\right)\right)\right)\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
Y\!\left(\iota_W\!\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\eta\!\left(\pi\sharp\!\left(\omega\right)\right)\right)\right)\right) \to Y\!\left(\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\eta\!\left(\pi\sharp\!\left(\omega\right)\right)\right)\right)\!\left(W\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\eta\!\left(\pi\sharp\!\left(\omega\right)\right)\right)\right)\!\left(W\right) \to \left(L_\pi\sharp(\omega)\!\left(\eta\right)\right)\!\left(W\right) - \left(L_\pi\sharp(\

In [8]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](ω, η)
 -> (L_π♯(ω)(η) + (-L_π♯(η)(ω)) + (-d(⟨π♯(ω), η⟩)))

[2] ⟨α, X⟩ → α(X) [axiom]
    ⟨π♯(ω), η⟩
 -> η(π♯(ω))

[3] MultiEval head linearity [axiom]
    (d(((-ι_X_ι_W(ω)(η)) + (ι_π♯(L_W(ω))(η) + (-ι_[W,π♯(ω)]_VF(η))))) + d(ι_W((L_π♯(ω)(η) + (-L_π♯(η)(ω)) + (-d(η(π♯(ω))))))))(Y)
 -> (d(((-ι_X_ι_W(ω)(η)) + (ι_π♯(L_W(ω))(η) + (-ι_[W,π♯(ω)]_VF(η)))))(Y) + d(ι_W((L_π♯(ω)(η) + (-L_π♯(η)(ω)) + (-d(η(π♯(ω)))))))(Y))

[4] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(((-ι_X_ι_W(ω)(η)) + (ι_π♯(L_W(ω))(η) + (-ι_[W,π♯(ω)]_VF(η)))))(Y)
 -> Y(((-ι_X_ι_W(ω)(η)) + (ι_π♯(L_W(ω))(η) + (-ι_[W,π♯(ω)]_VF(η)))))

[5] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_W((L_π♯(ω)(η) + (-L_π♯(η)(ω)) + (-d(η(π♯(ω)))))))(Y)
 -> Y(ι_W((L_π♯(ω)(η) + (-L_π♯(η)(ω)) + (-d(η(π♯(ω)))))))

[6] bare ι_X(ω) inside Act(D, _) → Act(D, Mul

## 3. Dual multivector side

### (1') $\tilde{D}^{\mathrm{SN}}_{\tilde{\mathcal{L}}_\eta}(U, V) = \tilde{\mathcal{L}}_{K_U \eta}\,V + \tilde{K}_{K_V \eta}\,U$

The left side is the obstruction
$\tilde{D}^{\mathrm{SN}}_{\tilde{\mathcal{L}}_\eta}(U,V)
= \tilde{\mathcal{L}}_\eta[U,V]_{\mathrm{SN}} - [\tilde{\mathcal{L}}_\eta U, V]_{\mathrm{SN}}
- [U, \tilde{\mathcal{L}}_\eta V]_{\mathrm{SN}}$;
on the right, two Lichnerowicz corrections tied to $K$ (the form
Cartan-remainder). The engine closes in 117 steps, via Lichnerowicz
expansion + SN/LBVF Sum-linearity + Sharp-Neg-linearity + the LBVF
Jacobi chain.


In [9]:
lhs = derivator(tilde_lie(eta, pi), default_sn, U, V)
K_U_eta = Act(K(U), eta)
K_V_eta = Act(K(V), eta)
rhs = Sum(
    Act(tilde_lie(K_U_eta, pi), V),
    Act(K_tilde(K_V_eta, pi), U),
)
chain = prob.prove_derivator(lhs, rhs, eval_args=(xi,), side="multivector")
print(f"(1') multivec, Dtilde^SN_{{Ltilde_eta}}(U,V) = Ltilde_{{K_U eta}} V + Ktilde_{{K_V eta}} U  -->  closed in {len(chain)} steps")
display_chain(chain)


(1') multivec, Dtilde^SN_{Ltilde_eta}(U,V) = Ltilde_{K_U eta} V + Ktilde_{K_V eta} U  -->  closed in 117 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left[U,\, V\right]_{[\cdot,\cdot]_{SN}} \to [U,V]_{VF} \quad \text{[[U, V]\_SN \ensuremath{\to} [U, V]\_VF  [both 1-vectors]]\,(axiom)} \\
\tilde{\mathcal{L}}_{\eta}\!\left([U,V]_{VF}\right) \to -X_{\eta([U,V]_{VF})} + \pi\sharp\!\left(L_{[U,V]_{VF}}\!\left(\eta\right)\right) - [[U,V]_{VF,\pi\sharp(\eta)]_{VF}} \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\alpha}(U) = \ensuremath{-}X\_{\ensuremath{\alpha}(U)} + \ensuremath{\pi}\ensuremath{\sharp}(L\_U \ensuremath{\alpha}) \ensuremath{-} [U, \ensuremath{\pi}\ensuremath{\sharp} \ensuremath{\alpha}]\_VF [\ensuremath{\pi}]]\,(axiom)} \\
\tilde{\mathcal{L}}_{\eta}\!\left(U\right) \to -X_\eta(U) + \pi\sharp\!\left(L_U\!\left(\eta\right)\right) - [U,\pi\sharp(\eta)]_{VF} \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\alpha}(U) = \ensuremath{-}X\_{\ensuremath{\alpha}(U)} + \ensuremath{\pi}\ensuremath{\sharp}(L\_U \ensuremath{\alpha}) \ensuremath{-} [U, \ensuremath{\pi}\ensuremath{\sharp} \ensuremath{\alpha}]\_VF [\ensuremath{\pi}]]\,(axiom)} \\
\left[-X_\eta(U) + \pi\sharp\!\left(L_U\!\left(\eta\right)\right) - [U,\pi\sharp(\eta)]_{VF},\, V\right]_{[\cdot,\cdot]_{SN}} \to \left[-X_\eta(U),\, V\right]_{[\cdot,\cdot]_{SN}} + \left[\pi\sharp\!\left(L_U\!\left(\eta\right)\right),\, V\right]_{[\cdot,\cdot]_{SN}} + \left[-[U,\pi\sharp(\eta)]_{VF},\, V\right]_{[\cdot,\cdot]_{SN}} \quad \text{[[Sum(a,b,\ensuremath{\dots}), Z]\_SN \ensuremath{\to} \ensuremath{\Sigma}\_i [a\_i, Z]\_SN  (and mirror)]\,(axiom)} \\
\left[-X_\eta(U),\, V\right]_{[\cdot,\cdot]_{SN}} \to -\left[X_\eta(U),\, V\right]_{[\cdot,\cdot]_{SN}} \quad \text{[[(\ensuremath{-}a), b]\_SN \ensuremath{\to} \ensuremath{-}[a, b]\_SN  /  [a, (\ensuremath{-}b)]\_SN \ensuremath{\to} \ensuremath{-}[a, b]\_SN]\,(axiom)} \\
\left[X_\eta(U),\, V\right]_{[\cdot,\cdot]_{SN}} \to [X_{\eta(U),V]_{VF}} \quad \text{[[U, V]\_SN \ensuremath{\to} [U, V]\_VF  [both 1-vectors]]\,(axiom)} \\
[X_{\eta(U),V]_{VF}} \to -[V,X_{\eta(U)]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
\left[\pi\sharp\!\left(L_U\!\left(\eta\right)\right),\, V\right]_{[\cdot,\cdot]_{SN}} \to [\pi\sharp(L_{U(\eta)),V]_{VF}} \quad \text{[[U, V]\_SN \ensuremath{\to} [U, V]\_VF  [both 1-vectors]]\,(axiom)} \\
[\pi\sharp(L_{U(\eta)),V]_{VF}} \to -[V,\pi\sharp(L_{U(\eta))]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
\left[-[U,\pi\sharp(\eta)]_{VF},\, V\right]_{[\cdot,\cdot]_{SN}} \to -\left[[U,\pi\sharp(\eta)]_{VF},\, V\right]_{[\cdot,\cdot]_{SN}} \quad \text{[[(\ensuremath{-}a), b]\_SN \ensuremath{\to} \ensuremath{-}[a, b]\_SN  /  [a, (\ensuremath{-}b)]\_SN \ensuremath{\to} \ensuremath{-}[a, b]\_SN]\,(axiom)} \\
\left[[U,\pi\sharp(\eta)]_{VF},\, V\right]_{[\cdot,\cdot]_{SN}} \to [[U,\pi\sharp(\eta)]_{VF,V]_{VF}} \quad \text{[[U, V]\_SN \ensuremath{\to} [U, V]\_VF  [both 1-vectors]]\,(axiom)} \\
[[U,\pi\sharp(\eta)]_{VF,V]_{VF}} \to -[V,[U,\pi\sharp(\eta)]_{VF]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
\tilde{\mathcal{L}}_{\eta}\!\left(V\right) \to -X_\eta(V) + \pi\sharp\!\left(L_V\!\left(\eta\right)\right) - [V,\pi\sharp(\eta)]_{VF} \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\alpha}(U) = \ensuremath{-}X\_{\ensuremath{\alpha}(U)} + \ensuremath{\pi}\ensuremath{\sharp}(L\_U \ensuremath{\alpha}) \ensuremath{-} [U, \ensuremath{\pi}\ensuremath{\sharp} \ensuremath{\alpha}]\_VF [\ensuremath{\pi}]]\,(axiom)} \\
\left[U,\, -X_\eta(V) + \pi\sharp\!\left(L_V\!\left(\eta\right)\right) - [V,\pi\sharp(\eta)]_{VF}\right]_{[\cdot,\cdot]_{SN}} \to \left[U,\, -X_\eta(V)\right]_{[\cdot,\cdot]_{SN}} + \left[U,\, \pi\sharp\!\left(L_V\!\left(\eta\right)\right)\right]_{[\cdot,\cdot]_{SN}} + \left[U,\, -[V,\pi\sharp(\eta)]_{VF}\right]_{[\cdot,\cdot]_{SN}} \quad \text{[[Sum(a,b,\ensuremath{\dots}), Z]\_SN \ensuremath{\to} \ensuremath{\Sigma}\_i [a\_i, Z]\_SN  (and mirror

In [10]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] [U, V]_SN → [U, V]_VF  [both 1-vectors] [axiom]
    [·,·]_SN(U, V)
 -> [U,V]_VF

[2] L̃_α(U) = −X_{α(U)} + π♯(L_U α) − [U, π♯ α]_VF [π] [axiom]
    L̃_η([U,V]_VF)
 -> ((-X_η([U,V]_VF)) + π♯(L_[U,V]_VF(η)) + (-[[U,V]_VF,π♯(η)]_VF))

[3] L̃_α(U) = −X_{α(U)} + π♯(L_U α) − [U, π♯ α]_VF [π] [axiom]
    L̃_η(U)
 -> ((-X_η(U)) + π♯(L_U(η)) + (-[U,π♯(η)]_VF))

[4] [Sum(a,b,…), Z]_SN → Σ_i [a_i, Z]_SN  (and mirror) [axiom]
    [·,·]_SN(((-X_η(U)) + π♯(L_U(η)) + (-[U,π♯(η)]_VF)), V)
 -> ([·,·]_SN((-X_η(U)), V) + [·,·]_SN(π♯(L_U(η)), V) + [·,·]_SN((-[U,π♯(η)]_VF), V))

[5] [(−a), b]_SN → −[a, b]_SN  /  [a, (−b)]_SN → −[a, b]_SN [axiom]
    [·,·]_SN((-X_η(U)), V)
 -> (-[·,·]_SN(X_η(U), V))

[6] [U, V]_SN → [U, V]_VF  [both 1-vectors] [axiom]
    [·,·]_SN(X_η(U), V)
 -> [X_η(U),V]_VF

[7] [Y, X]_VF → −[X, Y]_VF  [repr-canonical] [axiom]
    [X_η(U),V]_VF
 -> (-[V,X_η(U)]_VF)

[8] [U, V]_SN → [U, V]_VF  [both 1-vectors] [axiom]
    [·,·]_SN(π♯(L_U(η)), V)
 -> [π♯(L_U(η)),V]_VF

[9] [Y, X]_VF → −

### (2') $\tilde{\mathcal{L}}_{d\,\iota_U \eta}\,V = -[\tilde{d}\,\tilde{\iota}_\eta U,\;V]_{\mathrm{SN}}$

The direct dual of (2): Lichnerowicz expansion in the
$\tilde{\mathcal{L}}$ role; the SN bracket taken against $V$. Both
sides land on the Hamiltonian vector field $X_{\eta(U)}$, one (lhs)
through the SnBracket-on-1-forms route, giving an HVF with a $-$
sign; the other (rhs) from the $\tilde{d}_\pi$ application giving a
positive $\pi^\sharp \circ d$, the bridge
$\mathrm{Act}(\tilde{d}_\pi,\, f) \to \mathrm{Neg}(\mathrm{HVF}(f))$
unifies them.


In [11]:
head_lhs = Act(default_d, Act(interior(U), eta))
lhs = Act(tilde_lie(head_lhs, pi), V)
head_rhs = Act(tilde_d(pi), Act(tilde_interior(eta), U))
rhs = Neg(BracketApply(default_sn, head_rhs, V))
chain = prob.prove_derivator(lhs, rhs, eval_args=(xi,), side="multivector")
print(f"(2') multivec, Ltilde_{{d iota_U eta}} V = -[dtilde iotatilde_eta U, V]_SN  -->  closed in {len(chain)} steps")
display_chain(chain)


(2') multivec, Ltilde_{d iota_U eta} V = -[dtilde iotatilde_eta U, V]_SN  -->  closed in 25 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\tilde{\mathcal{L}}_{d\!\left(\iota_U\!\left(\eta\right)\right)}\!\left(V\right) \to -X_{d(\iota_U(\eta))(V)} + \pi\sharp\!\left(L_V\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right)\right) - [V,\pi\sharp(d(\iota_{U(\eta)))]_{VF}} \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\alpha}(U) = \ensuremath{-}X\_{\ensuremath{\alpha}(U)} + \ensuremath{\pi}\ensuremath{\sharp}(L\_U \ensuremath{\alpha}) \ensuremath{-} [U, \ensuremath{\pi}\ensuremath{\sharp} \ensuremath{\alpha}]\_VF [\ensuremath{\pi}]]\,(axiom)} \\
L_V\!\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right) \to d\!\left(L_V\!\left(\iota_U\!\left(\eta\right)\right)\right) \quad \text{[L\_X \ensuremath{\circ} d = d \ensuremath{\circ} L\_X (tilde, any mode)]\,(axiom)} \\
\pi\sharp\!\left(d\!\left(L_V\!\left(\iota_U\!\left(\eta\right)\right)\right)\right) \to X_{L_{V(\iota_U(\eta))}} \quad \text{[\ensuremath{\pi}\ensuremath{\sharp}(df) = X\_f [\ensuremath{\pi}]]\,(axiom)} \\
[V,\pi\sharp(d(\iota_{U(\eta)))]_{VF}} \to [V,X_{\eta(U)]_{VF}} \quad \text{[bare-atom-slot lift]\,(axiom)} \\
\left(-X_{d(\iota_U(\eta))(V)} + X_{L_{V(\iota_U(\eta))}} - [V,X_{\eta(U)]_{VF}}\right)\!\left(\xi\right) \to -X_{d(\iota_U(\eta))(V)}\!\left(\xi\right) + X_{L_{V(\iota_U(\eta))}}\!\left(\xi\right) - [V,X_{\eta(U)]_{VF}}\!\left(\xi\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
X_{d(\iota_U(\eta))(V)}\!\left(\xi\right) \to \xi\!\left(X_{d(\iota_U(\eta))(V)}\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pairing flip]]\,(axiom)} \\
\xi\!\left(X_{d(\iota_U(\eta))(V)}\right) \to -\left(\pi\sharp\!\left(\xi\right)\right)\!\left(\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right)\!\left(V\right)\right) \quad \text{[\ensuremath{\alpha}(X\_f) = \ensuremath{-}\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\alpha})(f) [\ensuremath{\pi}]]\,(axiom)} \\
\left(d\!\left(\iota_U\!\left(\eta\right)\right)\right)\!\left(V\right) \to V\!\left(\iota_U\!\left(\eta\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
V\!\left(\iota_U\!\left(\eta\right)\right) \to V\!\left(\eta\!\left(U\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))  [permissive]]\,(axiom)} \\
X_{L_{V(\iota_U(\eta))}}\!\left(\xi\right) \to \xi\!\left(X_{L_{V(\iota_U(\eta))}}\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pairing flip]]\,(axiom)} \\
\xi\!\left(X_{L_{V(\iota_U(\eta))}}\right) \to -\left(\pi\sharp\!\left(\xi\right)\right)\!\left(L_V\!\left(\iota_U\!\left(\eta\right)\right)\right) \quad \text{[\ensuremath{\alpha}(X\_f) = \ensuremath{-}\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\alpha})(f) [\ensuremath{\pi}]]\,(axiom)} \\
\left(\pi\sharp\!\left(\xi\right)\right)\!\left(L_V\!\left(\iota_U\!\left(\eta\right)\right)\right) \to \left(\pi\sharp\!\left(\xi\right)\right)\!\left(V\!\left(\iota_U\!\left(\eta\right)\right)\right) \quad \text{[Act(D, L\_X(f)) \ensuremath{\to} Act(D, X(f))  [D vf-like, f 0-form]]\,(axiom)} \\
V\!\left(\iota_U\!\left(\eta\right)\right) \to V\!\left(\eta\!\left(U\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))  [permissive]]\,(axiom)} \\
[V,X_{\eta(U)]_{VF}}\!\left(\xi\right) \to \xi\!\left([V,X_{\eta(U)]_{VF}}\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pairing flip]]\,(axiom)} \\
\tilde{d}\!\left(\tilde{\iota}_{\eta}\!\left(U\right)\right) \to \tilde{d}\!\left(U\!\left(\eta\right)\right) \quad \text{[bar

In [12]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L̃_α(U) = −X_{α(U)} + π♯(L_U α) − [U, π♯ α]_VF [π] [axiom]
    L̃_d(ι_U(η))(V)
 -> ((-X_d(ι_U(η))(V)) + π♯(L_V(d(ι_U(η)))) + (-[V,π♯(d(ι_U(η)))]_VF))

[2] L_X ∘ d = d ∘ L_X (tilde, any mode) [axiom]
    L_V(d(ι_U(η)))
 -> d(L_V(ι_U(η)))

[3] π♯(df) = X_f [π] [axiom]
    π♯(d(L_V(ι_U(η))))
 -> X_L_V(ι_U(η))

[4] bare-atom-slot lift [axiom]
    [V,π♯(d(ι_U(η)))]_VF
 -> [V,X_η(U)]_VF

[5] MultiEval head linearity [axiom]
    ((-X_d(ι_U(η))(V)) + X_L_V(ι_U(η)) + (-[V,X_η(U)]_VF))(ξ)
 -> ((-X_d(ι_U(η))(V)(ξ)) + X_L_V(ι_U(η))(ξ) + (-[V,X_η(U)]_VF(ξ)))

[6] X(ω) → ω(X)  [arity-1 covector pairing flip] [axiom]
    X_d(ι_U(η))(V)(ξ)
 -> ξ(X_d(ι_U(η))(V))

[7] α(X_f) = −π♯(α)(f) [π] [axiom]
    ξ(X_d(ι_U(η))(V))
 -> (-π♯(ξ)(d(ι_U(η))(V)))

[8] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_U(η))(V)
 -> V(ι_U(η))

[9] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X))  [permissive] [axiom]
    V(ι_U(η))
 -> V(η(U))

[10] X(ω) → ω(X) 

### (3') $0 = \tilde{d}\,\tilde{\iota}_{\mathcal{L}_U \eta}\,V - \tilde{d}\,\tilde{\iota}_{d\,\iota_V \eta}\,U + \tilde{d}\,\tilde{\iota}_\eta\,[U, V]_{\mathrm{SN}}$

The direct dual of (3): again a zero-sum, with the first two terms
as $\mathrm{lhs}_{\mathrm{pair}}$ and the third (positive) as
$\mathrm{rhs}_{\mathrm{third}}$.


In [13]:
v1p = Act(lie_derivative(U), eta)
v2p = Act(default_d, Act(interior(V), eta))
bracket_uv = BracketApply(default_sn, U, V)

lhs_pair = Sum(
    Act(tilde_d(pi), Act(tilde_interior(v1p), V)),
    Act(tilde_d(pi), Act(tilde_interior(eta), bracket_uv)),
)
rhs_third = Act(tilde_d(pi), Act(tilde_interior(v2p), U))
chain = prob.prove_derivator(lhs_pair, rhs_third, eval_args=(xi,), side="multivector")
print(f"(3') multivec, zero-sum reformulated lhs_pair = rhs_third  -->  closed in {len(chain)} steps")
display_chain(chain)


(3') multivec, zero-sum reformulated lhs_pair = rhs_third  -->  closed in 23 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\tilde{d}\!\left(\tilde{\iota}_{L_U\!\left(\eta\right)}\!\left(V\right)\right) \to \tilde{d}\!\left(V\!\left(L_U\!\left(\eta\right)\right)\right) \quad \text{[bare \ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega}(V) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(V, \ensuremath{\omega}))  (V deg 1)]\,(axiom)} \\
V\!\left(L_U\!\left(\eta\right)\right) \to \left(L_U\!\left(\eta\right)\right)\!\left(V\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pairing flip]]\,(axiom)} \\
\left(L_U\!\left(\eta\right)\right)\!\left(V\right) \to U\!\left(\eta\!\left(V\right)\right) - \eta\!\left([U,V]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\tilde{d}\!\left(U\!\left(\eta\!\left(V\right)\right) - \eta\!\left([U,V]_{VF}\right)\right) \to -X_{(U(\eta(V)) + (-\eta([U,V]_{VF})))} \quad \text{[Act(\ensuremath{\tilde{d}}\_\ensuremath{\pi}, f) \ensuremath{\to} \ensuremath{-}X\_f  [geometer sign]]\,(axiom)} \\
\left[U,\, V\right]_{[\cdot,\cdot]_{SN}} \to [U,V]_{VF} \quad \text{[[U, V]\_SN \ensuremath{\to} [U, V]\_VF  [both 1-vectors]]\,(axiom)} \\
\tilde{d}\!\left(\tilde{\iota}_{\eta}\!\left([U,V]_{VF}\right)\right) \to \tilde{d}\!\left([U,V]_{VF}\!\left(\eta\right)\right) \quad \text{[Act(D, \ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\alpha}([U,V]\_VF)) \ensuremath{\to} Act(D, MultiEval([U,V]\_VF, \ensuremath{\alpha}, covector))]\,(axiom)} \\
[U,V]_{VF}\!\left(\eta\right) \to \eta\!\left([U,V]_{VF}\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pairing flip]]\,(axiom)} \\
\tilde{d}\!\left(\eta\!\left([U,V]_{VF}\right)\right) \to -X_{\eta([U,V]_{VF})} \quad \text{[Act(\ensuremath{\tilde{d}}\_\ensuremath{\pi}, f) \ensuremath{\to} \ensuremath{-}X\_f  [geometer sign]]\,(axiom)} \\
\left(-X_{(U(\eta(V)) + (-\eta([U,V]_{VF})))} - X_{\eta([U,V]_{VF})}\right)\!\left(\xi\right) \to -X_{(U(\eta(V)) + (-\eta([U,V]_{VF})))}\!\left(\xi\right) - X_{\eta([U,V]_{VF})}\!\left(\xi\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
X_{(U(\eta(V)) + (-\eta([U,V]_{VF})))}\!\left(\xi\right) \to \xi\!\left(X_{(U(\eta(V)) + (-\eta([U,V]_{VF})))}\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pairing flip]]\,(axiom)} \\
\xi\!\left(X_{(U(\eta(V)) + (-\eta([U,V]_{VF})))}\right) \to -\left(\pi\sharp\!\left(\xi\right)\right)\!\left(U\!\left(\eta\!\left(V\right)\right) - \eta\!\left([U,V]_{VF}\right)\right) \quad \text{[\ensuremath{\alpha}(X\_f) = \ensuremath{-}\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\alpha})(f) [\ensuremath{\pi}]]\,(axiom)} \\
X_{\eta([U,V]_{VF})}\!\left(\xi\right) \to \xi\!\left(X_{\eta([U,V]_{VF})}\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pairing flip]]\,(axiom)} \\
\xi\!\left(X_{\eta([U,V]_{VF})}\right) \to -\left(\pi\sharp\!\left(\xi\right)\right)\!\left(\eta\!\left([U,V]_{VF}\right)\right) \quad \text{[\ensuremath{\alpha}(X\_f) = \ensuremath{-}\ensuremath{\pi}\ensuremath{\sharp}(\ensuremath{\alpha})(f) [\ensuremath{\pi}]]\,(axiom)} \\
\tilde{d}\!\left(\tilde{\iota}_{d\!\left(\iota_V\!\left(\eta\right)\right)}\!\left(U\right)\right) \to \tilde{d}\!\left(U\!\left(d\!\left(\iota_V\!\left(\eta\right)\right)\right)\right) \quad \text{[bare \ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega}(V) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(V, \ensuremath{\omega}))  (V deg 1)]\,(axiom)} \\
U\!\left(d\!\left(\iota_V\!\left(\eta\right)\right)\right) \to \left(d\!\left(\iota_V\!\left(\eta\right)\right)\right)\!\left(U\right) \quad \text{[X(\ensuremath{\omega}) \ensuremath{\to} \ensuremath{\omega}(X)  [arity-1 covector pa

In [14]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] bare ι̃_ω(V) inside Act(D, _) → Act(D, MultiEval(V, ω))  (V deg 1) [axiom]
    d̃_π(ι̃_L_U(η)(V))
 -> d̃_π(V(L_U(η)))

[2] X(ω) → ω(X)  [arity-1 covector pairing flip] [axiom]
    V(L_U(η))
 -> L_U(η)(V)

[3] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_U(η)(V)
 -> (U(η(V)) + (-η([U,V]_VF)))

[4] Act(d̃_π, f) → −X_f  [geometer sign] [axiom]
    d̃_π((U(η(V)) + (-η([U,V]_VF))))
 -> (-X_(U(η(V)) + (-η([U,V]_VF))))

[5] [U, V]_SN → [U, V]_VF  [both 1-vectors] [axiom]
    [·,·]_SN(U, V)
 -> [U,V]_VF

[6] Act(D, ι̃_α([U,V]_VF)) → Act(D, MultiEval([U,V]_VF, α, covector)) [axiom]
    d̃_π(ι̃_η([U,V]_VF))
 -> d̃_π([U,V]_VF(η))

[7] X(ω) → ω(X)  [arity-1 covector pairing flip] [axiom]
    [U,V]_VF(η)
 -> η([U,V]_VF)

[8] Act(d̃_π, f) → −X_f  [geometer sign] [axiom]
    d̃_π(η([U,V]_VF))
 -> (-X_η([U,V]_VF))

[9] MultiEval head linearity [axiom]
    ((-X_(U(η(V)) + (-η([U,V]_VF)))) + (-X_η([U,V]_VF)))(ξ)
 -> ((-X_(U(η(V)) + (-η([U,V]_VF)))(ξ)) + (-X_η(

## Conclusion

All six §3.1.5 identities, form (1)/(2)/(3) and dual
(1')/(2')/(3'), close syntactically via a single
`prob.prove_derivator(...)` call. Step counts: 109 / 21 / 30 (form)
and 117 / 25 / 23 (multivector).

**Form side:**

- (1): $D^{T^{*}M}_{\mathcal{L}_U}(\eta, \mu) = \mathcal{L}_{\tilde{K}_\eta U}\mu + K_{\tilde{K}_\mu U}\eta$, eval $(Y)$
- (2): $\mathcal{L}_{\tilde{d}\tilde{\iota}_\eta U}\mu = -[d\iota_U\eta, \mu]_K$, eval $(Y)$
- (3): $\sum d\iota_\bullet \cdot = 0$, eval $(Y)$

**Multivector side:**

- (1'): $\tilde{D}^{\mathrm{SN}}_{\tilde{\mathcal{L}}_\eta}(U, V) = \tilde{\mathcal{L}}_{K_U\eta}V + \tilde{K}_{K_V\eta}U$, eval $(\xi)$
- (2'): $\tilde{\mathcal{L}}_{d\iota_U\eta}V = -[\tilde{d}\tilde{\iota}_\eta U, V]_{\mathrm{SN}}$, eval $(\xi)$
- (3'): $\sum \tilde{d}\tilde{\iota}_\bullet \cdot = 0$, eval $(\xi)$

Phase 15.C had to bring three things on stage: form-side
Cartan-magic + the tilde Lichnerowicz expansion (for vector fields),
$\pi^\sharp$ Neg-linearity, and the slot-lift for opaque operator
atoms like `LieBracketVF`. Without these three, in particular (1')
cannot reach its 117-step closure, every step is readable as LaTeX
together with its `Definition` label via `display_chain(chain)`.
